# Ghost-Tip Protocol & Web3 dApp Wallet Master Plan

This document serves as the self-contained engineering blueprint and R&D output for building a privacy-first Web3 dApp wallet. It details the implementation plan for the **Ghost-Tip Protocol**: a stateless, untraceable eCash system for creator micropayments on the EVM.

This plan integrates the multiplicative blinding logic from the **BM-BLS (Blind Multisignature based on BLS)** paper, scaled down to a 1-of-1 signer configuration. The system operates strictly on **0.01 Sepolia ETH** denominations, features a Python-based off-chain mint, and defines a completely stateless web dApp where the blinding factor and all secrets are deterministically derived.

> **Note on Delivery Channel:** As per the current iteration, we are strictly relying on mathematical blinding to protect the signature in transit. The Mint broadcasts the blinded signature ($S'$) in plaintext back to the chain. Because it is blinded by $r$, third parties cannot unblind it or steal it without the user's deterministic seed.

---

## 1. Cryptographic Foundation: BM-BLS on EVM (BN254)

Ethereum’s native precompiles only support the **BN254 (alt_bn128)** curve. For the BLS blind signature, the protocol uses standard multiplicative blinding in $\mathbb{Z}_q$ (where $q$ is the order of the BN254 curve).

> **CRITICAL: Deterministic Derivation**
> To ensure the dApp remains completely stateless and recoverable from just a seed phrase, **all** cryptographic elements for a given token—including the blinding factor—are deterministically derived from the user's Master Seed and a `token_index`.

1. **Token Secret ($t$):** The Ethereum Address of the deterministic `Spend Keypair` (`secp256k1`).
2. **Blinding Factor ($r$):** A deterministic scalar $r\in\mathbb{Z}_q$ derived from the seed (e.g., `Keccak256(Seed || "blind" || index)`).
3. **Blinding:** The dApp maps $t$ to $G_1$: $Y=H_{G1}(t)$. It then multiplies by the deterministic blinding factor: $B=r\times Y$.
4. **Signing:** The Mint multiplies $B$ by its secret key $sk$: $S'=sk\times B$.
5. **Unblinding:** The dApp recalculates $r$, computes the modular inverse $r^{-1}\pmod q$, and unblinds: $S=r^{-1}\times S'$.
6. **Verification:** The Smart Contract verifies $e(S,G_2)==e(H_{G1}(t),PK_{mint})$.

---

## 2. Component 1: On-Chain Contracts (GhostVault.sol)

The Smart Contract acts as the **Ingress Vault** (accepting ETH and logging mint requests), the **Egress Vault** (verifying BLS pairings and dispensing ETH), and the decentralized messaging bus for the blinded signature delivery.

### State Variables

* `uint256[4] public PK_mint;` (The Mint's public key on the $G_2$ curve)
* `mapping(address => bool) public spentNullifiers;` (Tracks spent tokens using the Spend Address as the nullifier)
* `uint256 public constant DENOMINATION = 0.01 ether;`

### Function Interfaces

#### 1. `deposit` (Ingress)

Accepts the 0.01 ETH and the blinded point from the client.

* **Input:** `uint256[2] calldata blindedPointB` (The $x$ and $y$ coordinates of $B$ on $G_1$).
* **Logic:**
* `require(msg.value == DENOMINATION, "Must send exactly 0.01 ETH");`
* Emits `DepositLocked(uint256 depositId, uint256[2] B)`



#### 2. `announce` (Delivery / Registry)

Called exclusively by the Mint to deliver the blinded signature back to the user.

* **Input:** * `uint256 depositId`: Matches the ID from the `DepositLocked` event.
* `uint256[2] calldata blindedSignature`: The $x$ and $y$ coordinates of $S'$ on $G_1$.


* **Logic:** Emits `MintFulfilled(depositId, blindedSignature)`

#### 3. `redeem` (Egress)

Verifies the token and dispenses funds to the chosen recipient.

* **Input:**
* `address recipient`: The destination address for the 0.01 ETH.
* `bytes calldata spendSignature`: The ECDSA signature over `"Pay to: <recipient>"` (MEV protection).
* `uint256[2] calldata unblindedSignatureS`: The $x$ and $y$ coordinates of $S$ on $G_1$.


* **Logic:**
* `address nullifier = ecrecover(keccak256("Pay to: ", recipient), spendSignature);`
* `require(!spentNullifiers[nullifier], "Token spent");`
* `spentNullifiers[nullifier] = true;`
* `uint256[2] memory Y = hashToCurve(nullifier);`
* `require(ecPairing(unblindedSignatureS, G2, Y, PK_mint), "Invalid BLS Signature");`
* Transfers `0.01 ETH` to `recipient`.



### Smart Contract Implementation

```solidity
pragma solidity ^0.8.19;

contract GhostVault {
    mapping(address => bool) public spentNullifiers;
    uint256[4] public PK_mint;
    uint256 public constant DENOMINATION = 0.01 ether;

    event DepositLocked(uint256 indexed depositId, uint256[2] B);
    event MintFulfilled(uint256 indexed depositId, uint256[2] blindedSignature);

    // 1. Ingress
    function deposit(uint256[2] calldata blindedPointB) external payable {
        require(msg.value == DENOMINATION, "Must send exactly 0.01 ETH");
        uint256 depositId = uint256(keccak256(abi.encodePacked(msg.sender, block.timestamp, blindedPointB)));
        emit DepositLocked(depositId, blindedPointB);
    }

    // 2. Delivery / Registry
    function announce(uint256 depositId, uint256[2] calldata blindedSignature) external {
        // In a production environment, this could be restricted to authorized relayers or the Mint
        emit MintFulfilled(depositId, blindedSignature);
    }

    // 3. Egress
    function redeem(
        address recipient,
        bytes calldata spendSignature,
        uint256[2] calldata unblindedSignatureS
    ) external {
        // Extract nullifier natively via ecrecover
        bytes32 txHash = keccak256(abi.encodePacked("Pay to: ", recipient));
        address nullifier = recoverSigner(txHash, spendSignature);
        require(nullifier != address(0), "Invalid ECDSA signature");

        // Strict Double-Spend Check
        require(!spentNullifiers[nullifier], "Token already spent");
        spentNullifiers[nullifier] = true;

        // Map to BN254 and verify BLS Pairing
        uint256[2] memory Y = hashToCurve(nullifier);
        require(ecPairing(unblindedSignatureS, G2, Y, PK_mint), "Invalid BLS Signature");

        // Dispense funds
        payable(recipient).transfer(DENOMINATION);
    }

    // Helper functions for recoverSigner, hashToCurve, and ecPairing would be implemented here
}

```

---

## 3. Component 2: Off-Chain Mint (Python Daemon)

The Mint is a lightweight, stateless Python application. It monitors the blockchain, performs scalar multiplication on BN254, and posts the blinded signature back to the contract.

### Configuration / Secrets

* `MINT_BLS_PRIVKEY`: Scalar integer for the BN254 curve (used with `py_ecc.bn128`).

### Daemon Implementation

In [ ]:
import asyncio
from web3 import Web3
from py_ecc.bn128 import multiply
from mock_crypto import parse_g1_point  # Conceptual helper

w3 = Web3(Web3.WebsocketProvider('wss://sepolia.infura.io/ws/v3/YOUR-PROJECT-ID'))
MINT_BLS_PRIVKEY = load_bls_secret()

async def listen_for_deposits():
    """Background task to listen for DepositLocked events."""
    contract = w3.eth.contract(address="0x...", abi=GHOST_VAULT_ABI)
    event_filter = contract.events.DepositLocked.create_filter(fromBlock='latest')

    while True:
        for event in event_filter.get_new_entries():
            deposit_id = event['args']['depositId']
            blinded_point_b = event['args']['B']

            s_prime_coords = process_signature(blinded_point_b)
            submit_announcement(deposit_id, s_prime_coords)
        await asyncio.sleep(2)

def process_signature(blinded_point_b: list) -> list:
    """Performs blind signing (sk * B)."""
    # 1. Map to py_ecc G1 Point
    B = parse_g1_point(blinded_point_b)

    # 2. S' = sk * B
    S_prime = multiply(B, MINT_BLS_PRIVKEY)

    # 3. Return coordinates [x, y] for Solidity
    return [S_prime[0].n, S_prime[1].n]

def submit_announcement(deposit_id: int, s_prime_coords: list):
    """Submits the MintFulfilled transaction to the blockchain."""
    contract = w3.eth.contract(address="0x...", abi=GHOST_VAULT_ABI)
    tx = contract.functions.announce(deposit_id, s_prime_coords).build_transaction({
        'from': MINT_PUBLIC_ADDRESS,
        'nonce': w3.eth.get_transaction_count(MINT_PUBLIC_ADDRESS)
    })

    signed_tx = w3.eth.account.sign_transaction(tx, private_key=MINT_WALLET_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed_tx.rawTransaction)
    print(f"Announcement submitted: {tx_hash.hex()}")

---

## 4. Component 3: Frontend App (Web-Based dApp)

The browser-based dApp acts as the user's wallet. It holds the `Master Seed` in local browser memory and generates all state dynamically. It relies on standard JS cryptography (`noble-curves` for `secp256k1` and a WASM wrapper like `mcl-wasm` for BN254 math).

### Core Interfaces & Logic Implementation

```typescript
import { keccak256 } from 'ethereum-cryptography/keccak';
import { secp256k1 } from '@noble/curves/secp256k1';
import { multiplyBN254, hashToCurveBN254, modularInverse, verifyPairingBN254 } from './mock-bn254-wasm';

interface TokenSecrets {
    spendPriv: Uint8Array;
    spendAddress: string;
    r: bigint; // Scalar blinding factor
}

export class GhostTipClient {
    private masterSeed: Uint8Array;
    private mintBlsPubG2: Uint8Array;    // BN254 G2 (For BLS Verification)

    constructor(masterSeed: Uint8Array, mintBlsPubG2: Uint8Array) {
        this.masterSeed = masterSeed;
        this.mintBlsPubG2 = mintBlsPubG2;
    }

    // 1. Deterministic State Derivation
    deriveTokenSecrets(tokenIndex: number): TokenSecrets {
        const indexBytes = new Uint8Array(new Uint32Array([tokenIndex]).buffer);
        const baseMaterial = keccak256(new Uint8Array([...this.masterSeed, ...indexBytes]));

        // Spend Keypair (Identity / Nullifier)
        const spendPriv = keccak256(new Uint8Array([...Buffer.from("spend"), ...baseMaterial]));
        const spendAddress = generateEthereumAddress(secp256k1.getPublicKey(spendPriv));

        // Blinding Factor (r)
        const rBytes = keccak256(new Uint8Array([...Buffer.from("blind"), ...baseMaterial]));
        const r = BigInt('0x' + Buffer.from(rBytes).toString('hex')) % BN254_ORDER;

        return { spendPriv, spendAddress, r };
    }

    // 2. Minting Request Construction
    createMintTx(tokenIndex: number) {
        const { spendAddress, r } = this.deriveTokenSecrets(tokenIndex);
        
        // Y = H_G(Address) -> B = r * Y
        const Y = hashToCurveBN254(spendAddress);
        const B = multiplyBN254(Y, r);
        
        return {
            value: "10000000000000000", // 0.01 ETH in Wei
            data: encodeDepositCall(B)  // ABI encoding for GhostVault.deposit(B)
        };
    }

    // 3. Stateless Recovery
    async scanAndRecoverTokens(startIndex: number, endIndex: number) {
        // Fetch MintFulfilled events containing blindedSignature (S')
        const events = await fetchMintFulfilledEventsFromRPC();
        const availableTokens = [];

        for (let i = startIndex; i <= endIndex; i++) {
            const { r, spendAddress, spendPriv } = this.deriveTokenSecrets(i);
            
            // To find our token, we actually need to know which DepositID maps to our index.
            // In a real implementation, we'd cross-reference the DepositLocked(B) event
            // that matches our deterministic B point to find the exact event.
            // Assuming `event` here represents the matching fulfillment for index `i`:
            
            const matchEvent = findMatchingFulfillmentForIndex(i, events, this);
            if (!matchEvent) continue;

            const sPrime = matchEvent.blindedSignature;
            
            // Unblind: S = S' * r^-1
            const rInv = modularInverse(r, BN254_ORDER);
            const S = multiplyBN254(sPrime, rInv);
            
            // LOCAL CRYPTOGRAPHIC VERIFICATION
            // We must verify: e(S, G2) == e(H(SpendAddress), MintBlsPubG2)
            const Y = hashToCurveBN254(spendAddress);
            const isValidSignature = verifyPairingBN254(S, Y, this.mintBlsPubG2);
            
            if (!isValidSignature) {
                console.warn(`Token ${i} unblinded, but BLS signature is mathematically invalid!`);
                continue;
            }
            
            // Check if already spent on-chain
            const isSpent = await checkSpentNullifierOnChain(spendAddress);
            if (!isSpent) {
                availableTokens.push({ index: i, spendPriv, signatureS: S });
            }
        }
        return availableTokens;
    }

    // 4. Redemption Proof Generation
    generateRedemptionPayload(tokenIndex: number, destinationAddress: string, unblindedS: any) {
        const { spendPriv } = this.deriveTokenSecrets(tokenIndex);
        
        // Sign MEV protection payload
        const msgHash = keccak256(Buffer.from(`Pay to: ${destinationAddress}`));
        const spendSignature = secp256k1.sign(msgHash, spendPriv).toCompactRawBytes();
        
        // This payload is submitted to GhostVault.redeem()
        return {
            recipient: destinationAddress,
            spendSignature: spendSignature,
            unblindedSignatureS: unblindedS
        };
    }
}

```

---

## Appendix: Key Generation Instructions for Ivan

Ivan's primary setup task is configuring the off-chain mint's cryptography. Since the ECDH view key architecture has been removed, the Mint requires exactly **one** private key: the scalar for the BN254 curve to perform the mathematical blind signing ($S' = sk \times B$).

### The BN254 (alt_bn128) BLS Keypair Script

The public key on the $G_2$ curve ($PK_{mint}$) must be exported in two different formats: one for the EVM precompiles, and one for the frontend WASM library. Ivan can run this Python script to generate and format everything cleanly.

In [ ]:
import os
from py_ecc.bn128 import curve_order, G2, multiply

def generate_keys():
    print("========================================")
    print("🔒 GHOST-TIP MINT KEY GENERATION UTILITY")
    print("========================================\n")

    # Generate a random scalar within the curve's order
    sk_bytes = os.urandom(32)
    sk_int = int.from_bytes(sk_bytes, 'big') % curve_order

    # Calculate PK_mint = sk * G2
    pk_g2 = multiply(G2, sk_int)

    # Extract coordinates
    # py_ecc G2 points are tuples of FQ2 polynomials: ((x.real, x.imag), (y.real, y.imag))
    # Note: Solidity ecPairing expects: [x.imag, x.real, y.imag, y.real]
    x_real = pk_g2[0].coeffs[0].n
    x_imag = pk_g2[0].coeffs[1].n
    y_real = pk_g2[1].coeffs[0].n
    y_imag = pk_g2[1].coeffs[1].n

    print("--- BN254 BLS KEYS ---")
    print(f"MINT_BLS_PRIVKEY_INT      (Python .env) : {sk_int}\n")

    # Format for Solidity (Base-10 Integers)
    print("PK_MINT_SOLIDITY          (Smart Contract):")
    print(f"[{x_imag},\n {x_real},\n {y_imag},\n {y_real}]\n")

    # Format for TypeScript / mcl-wasm (Hex Strings)
    print("PK_MINT_TYPESCRIPT        (Frontend TS):")
    print(f"['{hex(x_imag)}',\n '{hex(x_real)}',\n '{hex(y_imag)}',\n '{hex(y_real)}']\n")

if __name__ == "__main__":
    generate_keys()

### Distribution Checklist

1. **For Ivan (Python Mint Server `/.env`)**: Save `MINT_BLS_PRIVKEY_INT` locally to perform $S' = sk \times B$. Add `MINT_WALLET_KEY` to pay for Sepolia gas.
2. **For Fabio (Smart Contracts `/contracts/GhostVault.sol`)**: Send the **`PK_MINT_SOLIDITY`** array. Fabio must copy this base-10 `uint256[4]` array into the contract's state so `ecPairing` can verify the unblinded tokens.
3. **For Simoneth (Frontend dApp `/src/config/sepolia.ts`)**: Send the **`PK_MINT_TYPESCRIPT`** array. Simoneth passes this array into the frontend's pairing function to locally verify the BLS math *before* attempting a redemption.